In [ ]:
%matplotlib inline

from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import vix_tcn_revin_xai_plus as vtrx
import metrics_over_time_v2 as metrics_mod
import cdew_concepts_v2 as concepts_mod
import event_warping as ew

CSV_PATH = ROOT / "data" / "raw" / "timeseries_data.csv"
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

In [ ]:
bundle_path = ROOT / "outputs" / "bundles" / "best_tcn_bundle.pt"
if not bundle_path.exists():
    bundle_path = ROOT / "outputs" / "bundles" / "best_model_bundle.pt"

model, bundle_meta, snapshot = vtrx.load_model_bundle(str(bundle_path), device=DEVICE)
cfg = vtrx.Config(**snapshot["cfg"])
cfg.csv_path = str(CSV_PATH)

target_mode = snapshot.get("target_mode", bundle_meta.get("target_mode", "level"))
print("bundle_path :", bundle_path)
print("target_mode :", target_mode)

In [ ]:
df_raw = vtrx.load_frame(cfg.csv_path, cfg.index_col, list(cfg.drop_cols))

_, _, _, meta = vtrx.build_dataloaders(
    df_raw=df_raw,
    target_col=cfg.target_col,
    seq_len=cfg.seq_len,
    batch_size=cfg.batch_size,
    train_ratio=cfg.train_ratio,
    val_ratio=cfg.val_ratio,
    num_workers=cfg.num_workers,
    pin_memory=(cfg.pin_memory and DEVICE.type == "cuda"),
    persistent_workers=cfg.persistent_workers,
    target_mode=target_mode,
)

In [ ]:
RUN_METRICS = True

In [ ]:
if RUN_METRICS:
    met_result = metrics_mod.run_metrics_over_time_v2(
        vtrx_module=vtrx,
        ew_module=ew,
        model=model,
        meta=meta,
        cfg=cfg,
        df_raw=df_raw,
        device=DEVICE,
        reference_mode="multi_ref",
        n_prototypes=5,
        band=5,
        k_dtdw=3,
        dist_method="wasserstein",
        alpha=1.5,
        top_p=0.20,
        weight_mode="local",
        gamma=1.0,
        horizon=5,
        q_event=0.98,
        evaluate_auc=True,
        evaluate_retrieval=True,
        k_list=[1, 5, 10, 20],
        alphas_sensitivity=[0.5, 1.0, 1.5, 2.0, 3.0],
        top_ps_sensitivity=[0.10, 0.15, 0.20, 0.30, 0.50, 1.0],
        save_dir=str(ROOT / "outputs" / "metrics"),
        show=True,
        seed=42,
        debug_max_n=None,
    )
else:
    raise RuntimeError("RUN_METRICS=False")

In [ ]:
metrics_dir = ROOT / "outputs" / "metrics"

auc_df = pd.read_csv(metrics_dir / "metric_auc.csv")
ret_df = pd.read_csv(metrics_dir / "retrieval_at_k.csv")
atp_df = pd.read_csv(metrics_dir / "alpha_top_p_sensitivity.csv")

print("=== metric_auc ===")
display(auc_df)

print("=== retrieval_at_k (ref_id=0) ===")
display(ret_df[ret_df["ref_id"] == 0])

print("=== alpha_top_p_sensitivity (top 10 by auc) ===")
display(atp_df.sort_values("auc", ascending=False).head(10))

In [ ]:
for img_name in [
    "metric_over_time.png",
    "metric_boxplots.png",
    "metric_auc_bar.png",
    "ref_sensitivity.png",
    "retrieval_at_k.png",
]:
    img_path = metrics_dir / img_name
    if img_path.exists():
        display(Image(filename=str(img_path), width=900))

In [ ]:
RUN_CDEW = True

In [ ]:
if RUN_CDEW:
    cdew_result = concepts_mod.run_cdew_analysis(
        vtrx_module=vtrx,
        ew_module=ew,
        model=model,
        meta=meta,
        cfg=cfg,
        df_raw=df_raw,
        device=DEVICE,
        Ea_all=met_result["Ea_all"],
        cam_all=met_result["cam_all"],
        raw_windows=met_result["raw_windows"],
        is_event=met_result["is_event"],
        label_dates=met_result["label_dates"],
        N=met_result["N"],
        refs=met_result["refs"],
        safe_asset_cols=("Gold", "미국채", "US10Y", "10Y", "Treasury"),
        risk_asset_col=("S&P", "S&P 500", "SP500", "SPX"),
        q_safe=0.90,
        threshold_source="train_only",
        tcav_cv_folds=5,
        band=5,
        normalize_cam=True,
        clip_cam_nonneg=True,
        n_perm=2000,
        fdr_correction=True,
        save_dir=str(ROOT / "outputs" / "cdew"),
        show=True,
        seed=42,
    )
else:
    raise RuntimeError("RUN_CDEW=False")

In [ ]:
cdew_dir = ROOT / "outputs" / "cdew"

print("=== cdew_effects ===")
display(pd.read_csv(cdew_dir / "cdew_effects.csv"))

print("=== tcav_cv_scores ===")
display(pd.read_csv(cdew_dir / "tcav_cv_scores.csv"))

print("=== cav_stability ===")
display(pd.read_csv(cdew_dir / "cav_stability.csv"))

In [ ]:
for img_name in [
    "cdew_over_time.png",
    "cdew_box_event.png",
    "cdew_box_concept.png",
    "tcav_cv_scores.png",
    "concept_weighted_example_pair.png",
]:
    img_path = cdew_dir / img_name
    if img_path.exists():
        display(Image(filename=str(img_path), width=900))

In [ ]:
# Multi-concept dashboard definitions

def _flight_to_safety(df_raw, df_ref):
    try:
        gold = concepts_mod._resolve_col(df_raw, ("Gold",))
        spx = concepts_mod._resolve_col(df_raw, ("SPX", "S&P", "S&P 500", "SP500"))
        gr = df_raw[gold].pct_change()
        sr = df_raw[spx].pct_change()
        thr = np.nanquantile(df_ref[gold].pct_change().dropna().values, 0.90)
        return (gr >= thr) & (sr < 0)
    except Exception:
        return pd.Series(False, index=df_raw.index)

def _inflation_shock(df_raw, df_ref):
    try:
        wti = concepts_mod._resolve_col(df_raw, ("WTI", "Oil", "CrudeOil", "Crude"))
        gold = concepts_mod._resolve_col(df_raw, ("Gold",))
        wr = df_raw[wti].pct_change()
        gr = df_raw[gold].pct_change()
        thr_w = np.nanquantile(df_ref[wti].pct_change().dropna().values, 0.95)
        return (wr >= thr_w) & (gr > 0)
    except Exception:
        return pd.Series(False, index=df_raw.index)

def _liquidity_squeeze(df_raw, df_ref):
    try:
        dxy = concepts_mod._resolve_col(df_raw, ("DXY", "Dollar", "USD"))
        vix = concepts_mod._resolve_col(df_raw, ("VIX",))
        dr = df_raw[dxy].pct_change()
        vr = df_raw[vix].pct_change()
        thr_d = np.nanquantile(df_ref[dxy].pct_change().dropna().values, 0.90)
        return (dr >= thr_d) & (vr > 0)
    except Exception:
        return pd.Series(False, index=df_raw.index)

def _emerging_market_shock(df_raw, df_ref):
    try:
        kospi = concepts_mod._resolve_col(df_raw, ("KOSPI", "KS11"))
        vix = concepts_mod._resolve_col(df_raw, ("VIX",))
        kr = df_raw[kospi].pct_change()
        vr = df_raw[vix].pct_change()
        thr_k = np.nanquantile(df_ref[kospi].pct_change().dropna().values, 0.05)
        return (kr <= thr_k) & (vr > 0)
    except Exception:
        return pd.Series(False, index=df_raw.index)

concept_definitions = {
    "FlightToSafety": _flight_to_safety,
    "InflationShock": _inflation_shock,
    "LiquiditySqueeze": _liquidity_squeeze,
    "EmergingMarketShock": _emerging_market_shock,
}